# Adaptive benchmark Qwen / GLM в ИФТ-контуре

Этот ноутбук **сначала эмпирически проверяет**, какие параметры реально принимает контурный API для `Qwen3.5-397b` и `glm-5.1`, включая разные варианты включения/выключения reasoning/thinking. После discovery он строит тестовую матрицу **только из реально принятых параметров** и ограничивает итоговое число конфигураций максимумом `MAX_TOTAL_CONFIGS = 12`.

Принципиально:
- промпты не меняются: используется только `system_prompt` + `user_prompt`;
- `llm_input_human_json` не используется;
- сначала один вопрос прогоняется по всем выбранным конфигам, потом следующий вопрос;
- если reasoning-параметры реально принимаются, они попадают в конфиги как отдельный фактор `reasoning on/off`.

Главный критерий отбора итоговых конфигураций — потенциальное влияние на время работы. Сначала notebook быстро показывает, какие параметры backend реально принимает. Затем финальная матрица выбирает параметры, которые по смыслу сильнее всего могут менять latency: `reasoning on/off`, `max_tokens`, sampling (`temperature`, `top_p`) и `repetition_penalty`, если они доступны. Если включить `RUN_LATENCY_IMPACT_DISCOVERY=True`, порядок отбора дополнительно будет опираться на измеренный latency-сдвиг на реальных prompts.

Во время benchmark notebook сохраняет промежуточные Excel-checkpoint-и каждые 3 вопроса: numbered-файл `checkpoint_after_XXXX_questions.xlsx` и свежую копию `checkpoint_latest.xlsx`.




## Как работает discovery

1. Сначала запускается быстрый **capability discovery** на коротком техническом prompt-е, а не на банковских данных. Его задача — быстро ответить на простой вопрос: какие параметры backend реально принимает для `Qwen3.5-397b` и `glm-5.1`.
2. После capability discovery notebook сразу выводит таблицу `AvailableParameters`: какие варианты параметров доступны, какие отвергнуты, и какой JSON patch соответствует каждому принятому варианту.
3. Только после этого строятся итоговые конфиги, максимум 12. Если доступны пары `reasoning off/on`, они включаются как важный фактор времени. Если доступны `max_tokens`, `temperature`, `top_p`, `repetition_penalty`, они используются как кандидаты для latency-ориентированной матрицы.
4. Опциональный дорогой `RUN_LATENCY_IMPACT_DISCOVERY` можно включить отдельно, если нужно до benchmark дополнительно оценить latency-сдвиги на реальных prompts. По умолчанию он выключен, чтобы не ждать 20 минут до простого списка доступных параметров.



In [ ]:
# =========================
# 0. Imports
# =========================

import csv
import hashlib
import json
import os
import random
import re
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", message="Unverified HTTPS request")
warnings.filterwarnings("ignore", category=UserWarning)

try:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
except Exception:
    pass

try:
    from gigachat import GigaChat
except Exception as exc:
    raise RuntimeError("Не удалось импортировать gigachat. Запускайте в окружении, где работал ift_agent_test.ipynb") from exc

RUN_STARTED_AT = datetime.now().strftime("%Y%m%d_%H%M%S")
print("Run started:", RUN_STARTED_AT)


In [ ]:
# =========================
# 1. Settings
# =========================

DATA_CSV = Path("max_inputs_first_200_unique_questions.csv")

GIGACHAT_API_URL = "https://gigachat-ift.sberdevices.delta.sbrf.ru/v1"
SCOPE = "GIGACHAT_API_PERS"
CERT_FILE = "45500_cert.pem"
KEY_FILE = "45500_key.pem"
VERIFY_SSL_CERTS = False
PROFANITY_CHECK = False

MODELS = {
    "qwen": "Qwen3.5-397b",
    "glm": "glm-5.1",
}

# Сначала быстрый capability discovery, потом построение конфигов и benchmark.
RUN_DISCOVERY = True
RUN_BENCHMARK = True

# Capability discovery проверяет только, принимает ли backend параметр.
# Он работает на коротком техническом prompt, а не на длинных банковских данных.
CAPABILITY_DISCOVERY_TIMEOUT_SEC = 60
CAPABILITY_SYSTEM_PROMPT = "Ты тестовый ассистент. Отвечай максимально кратко."
CAPABILITY_USER_PROMPT = "Ответь одним словом: OK"
STOP_AFTER_CAPABILITY_DISCOVERY = False

# Опционально можно отдельно померить latency-impact на реальных prompts, но по умолчанию выключено,
# чтобы перед отбором конфигов быстро увидеть, какие параметры вообще доступны.
RUN_LATENCY_IMPACT_DISCOVERY = False
LATENCY_DISCOVERY_ROWS_PER_STAGE = 3
LATENCY_DISCOVERY_TIMEOUT_SEC = 180

# Ограничение итоговой матрицы.
MAX_TOTAL_CONFIGS = 12
MAX_CONFIGS_PER_MODEL = max(1, MAX_TOTAL_CONFIGS // len(MODELS))

# Для включения reasoning в финальные конфиги достаточно, чтобы backend реально принял off/on параметры.
# Эффект reasoning затем проверяется уже в benchmark на настоящих prompts.
REQUIRE_OBSERVED_EFFECT_FOR_REASONING = False

# Полный запуск: full. Для проверки можно smoke/sample.
RUN_MODE = "sample"  # "full" | "smoke" | "sample"
SMOKE_ROWS_PER_STAGE = 2
SAMPLE_ROWS_TOTAL = 60
RANDOM_SEED = 42

MAX_WORKERS = 1
REQUEST_TIMEOUT_SEC = 270
MAX_RETRIES = 2
RETRY_BACKOFF_SEC = 8
JITTER_SEC = 2
USE_STREAMING = True
FALLBACK_TO_NON_STREAMING = True

CONFIG_ORDER_STRATEGY = "rotate"  # "fixed" | "rotate" | "shuffle_by_question"

SAVE_FULL_ANSWERS_TO_EXCEL = True
ANSWER_PREVIEW_CHARS = 500

# Checkpoint Excel: сохраняем промежуточный отчет каждые N вопросов, а не каждые N вызовов.
# При MAX_TOTAL_CONFIGS=12 это значит примерно 36 новых raw-записей между checkpoint-ами.
CHECKPOINT_EVERY_QUESTIONS = 3
SAVE_NUMBERED_CHECKPOINTS = True

OUTPUT_DIR = Path("benchmark_outputs") / f"qwen_glm_adaptive_{RUN_STARTED_AT}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_JSONL_PATH = OUTPUT_DIR / "raw_results.jsonl"
DISCOVERY_JSONL_PATH = OUTPUT_DIR / "discovery_results.jsonl"
EXCEL_REPORT_PATH = OUTPUT_DIR / "qwen_glm_adaptive_benchmark_report.xlsx"
CHECKPOINT_LATEST_PATH = OUTPUT_DIR / "checkpoint_latest.xlsx"

print("Output dir:", OUTPUT_DIR.resolve())






In [ ]:
# =========================
# 2. Load dataset
# =========================

csv.field_size_limit(sys.maxsize)
with DATA_CSV.open("r", encoding="utf-8-sig", newline="") as f:
    df_inputs = pd.DataFrame(list(csv.DictReader(f)))

required = ["type", "stage", "question", "system_prompt", "user_prompt"]
missing = [c for c in required if c not in df_inputs.columns]
if missing:
    raise ValueError(f"Нет обязательных колонок: {missing}")

if "llm_input_human_json" in df_inputs.columns:
    print("llm_input_human_json найден, но в запросы не используется.")

df_inputs = df_inputs.reset_index(drop=True).copy()
df_inputs["dataset_row_id"] = df_inputs.index.astype(int)
df_inputs["system_chars"] = df_inputs["system_prompt"].fillna("").str.len()
df_inputs["user_chars"] = df_inputs["user_prompt"].fillna("").str.len()
df_inputs["input_chars"] = df_inputs["system_chars"] + df_inputs["user_chars"]
df_inputs["old_output_chars"] = df_inputs.get("llm_output", pd.Series([""] * len(df_inputs))).fillna("").str.len()

display(df_inputs.groupby("type").agg(rows=("dataset_row_id", "count"), min_chars=("input_chars", "min"), p50_chars=("input_chars", "median"), max_chars=("input_chars", "max")).reset_index())


In [ ]:
# =========================
# 3. Generic helpers
# =========================

_client_cache: Dict[Tuple[Any, ...], GigaChat] = {}


def make_client(timeout: int) -> GigaChat:
    if MAX_WORKERS and MAX_WORKERS > 1:
        return GigaChat(
            base_url=GIGACHAT_API_URL,
            scope=SCOPE,
            timeout=timeout,
            profanity_check=PROFANITY_CHECK,
            cert_file=CERT_FILE,
            key_file=KEY_FILE,
            verify_ssl_certs=VERIFY_SSL_CERTS,
        )
    key = (GIGACHAT_API_URL, SCOPE, CERT_FILE, KEY_FILE, timeout, VERIFY_SSL_CERTS, PROFANITY_CHECK)
    if key not in _client_cache:
        _client_cache[key] = GigaChat(
            base_url=GIGACHAT_API_URL,
            scope=SCOPE,
            timeout=timeout,
            profanity_check=PROFANITY_CHECK,
            cert_file=CERT_FILE,
            key_file=KEY_FILE,
            verify_ssl_certs=VERIFY_SSL_CERTS,
        )
    return _client_cache[key]


def obj_to_dict(obj: Any) -> Any:
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, list):
        return [obj_to_dict(x) for x in obj]
    if isinstance(obj, tuple):
        return [obj_to_dict(x) for x in obj]
    if isinstance(obj, dict):
        return {str(k): obj_to_dict(v) for k, v in obj.items()}
    if hasattr(obj, "model_dump"):
        try:
            return obj.model_dump(by_alias=True, exclude_none=False)
        except TypeError:
            return obj.model_dump()
    if hasattr(obj, "dict"):
        try:
            return obj.dict(by_alias=True)
        except TypeError:
            return obj.dict()
    return str(obj)


def deep_get(data: Any, paths: List[List[Any]], default=None):
    for path in paths:
        cur = data
        ok = True
        for key in path:
            try:
                cur = cur[key] if isinstance(key, int) else cur.get(key)
            except Exception:
                ok = False
                break
            if cur is None:
                ok = False
                break
        if ok:
            return cur
    return default


def deep_merge(base: Dict[str, Any], patch: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(base)
    for k, v in (patch or {}).items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = deep_merge(out[k], v)
        else:
            out[k] = v
    return out


def text_hash(text: str) -> str:
    return hashlib.sha256((text or "").encode("utf-8")).hexdigest()[:16]


def extract_content(response_dict: Dict[str, Any]) -> str:
    v = deep_get(response_dict, [["choices", 0, "message", "content"]], "")
    return v if isinstance(v, str) else ""


def extract_finish_reason(response_dict: Dict[str, Any]) -> Optional[str]:
    return deep_get(response_dict, [["choices", 0, "finish_reason"], ["choices", 0, "finishReason"]])


def extract_reasoning_text(response_dict: Dict[str, Any]) -> str:
    candidates = [
        ["choices", 0, "message", "reasoning"],
        ["choices", 0, "message", "reasoning_content"],
        ["choices", 0, "message", "thinking"],
        ["choices", 0, "delta", "reasoning"],
        ["choices", 0, "delta", "reasoning_content"],
    ]
    v = deep_get(response_dict, candidates, "")
    return v if isinstance(v, str) else ""


def normalize_usage(usage: Any) -> Dict[str, Any]:
    d = obj_to_dict(usage) or {}
    return d if isinstance(d, dict) else {}


def extract_usage_numbers(usage: Dict[str, Any]) -> Dict[str, Any]:
    details = usage.get("completion_tokens_details") or usage.get("completionTokensDetails") or {}
    return {
        "prompt_tokens": usage.get("prompt_tokens") or usage.get("promptTokens"),
        "completion_tokens": usage.get("completion_tokens") or usage.get("completionTokens"),
        "total_tokens": usage.get("total_tokens") or usage.get("totalTokens"),
        "precached_prompt_tokens": usage.get("precached_prompt_tokens") or usage.get("precachedPromptTokens"),
        "reasoning_tokens": usage.get("reasoning_tokens") or usage.get("reasoningTokens") or details.get("reasoning_tokens") or details.get("reasoningTokens"),
    }


def count_words(text: str) -> int:
    return len(re.findall(r"[A-Za-zА-Яа-яЁё0-9_]+", text or ""))


In [ ]:
# =========================
# 4. Low-level LLM call using raw dict payload
# =========================

# Важно: используем dict payload, а не Chat-модель pydantic. Так мы можем реально попробовать поля,
# которых нет в Python-сигнатуре обертки, но которые может поддерживать backend.

def build_payload(system_prompt: str, user_prompt: str, model: str, base_params: Dict[str, Any], extra_params: Optional[Dict[str, Any]], stream: bool) -> Dict[str, Any]:
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "stream": bool(stream),
        "profanity_check": PROFANITY_CHECK,
    }
    payload = deep_merge(payload, {k: v for k, v in (base_params or {}).items() if v is not None})
    payload = deep_merge(payload, extra_params or {})
    return payload


def run_payload(system_prompt: str, user_prompt: str, model: str, base_params: Dict[str, Any], extra_params: Optional[Dict[str, Any]], timeout_sec: int, use_streaming: bool = USE_STREAMING) -> Dict[str, Any]:
    client = make_client(timeout_sec)
    start = time.perf_counter()
    first_chunk_at = None
    first_content_at = None
    chunks_seen = 0
    content_parts: List[str] = []
    reasoning_parts: List[str] = []
    last_chunk_dict: Dict[str, Any] = {}
    response_dict: Dict[str, Any] = {}
    usage_dict: Dict[str, Any] = {}

    if use_streaming:
        payload = build_payload(system_prompt, user_prompt, model, base_params, extra_params, stream=True)
        try:
            for chunk in client.stream(payload):
                now = time.perf_counter()
                chunks_seen += 1
                if first_chunk_at is None:
                    first_chunk_at = now
                chunk_dict = obj_to_dict(chunk) or {}
                last_chunk_dict = chunk_dict
                delta_content = deep_get(chunk_dict, [["choices", 0, "delta", "content"]], "")
                if isinstance(delta_content, str) and delta_content:
                    if first_content_at is None:
                        first_content_at = now
                    content_parts.append(delta_content)
                delta_reasoning = extract_reasoning_text(chunk_dict)
                if delta_reasoning:
                    reasoning_parts.append(delta_reasoning)
                if chunk_dict.get("usage"):
                    usage_dict = normalize_usage(chunk_dict.get("usage"))
        except Exception:
            if not FALLBACK_TO_NON_STREAMING:
                raise
            payload = build_payload(system_prompt, user_prompt, model, base_params, extra_params, stream=False)
            response = client.chat(payload)
            response_dict = obj_to_dict(response) or {}
            usage_dict = normalize_usage(response_dict.get("usage"))
            content = extract_content(response_dict)
            reasoning = extract_reasoning_text(response_dict)
            content_parts = [content] if content else []
            reasoning_parts = [reasoning] if reasoning else []
    else:
        payload = build_payload(system_prompt, user_prompt, model, base_params, extra_params, stream=False)
        response = client.chat(payload)
        response_dict = obj_to_dict(response) or {}
        usage_dict = normalize_usage(response_dict.get("usage"))
        content = extract_content(response_dict)
        reasoning = extract_reasoning_text(response_dict)
        content_parts = [content] if content else []
        reasoning_parts = [reasoning] if reasoning else []

    end = time.perf_counter()
    answer = "".join(content_parts)
    reasoning_text = "".join(reasoning_parts)
    finish_reason = extract_finish_reason(response_dict) or extract_finish_reason(last_chunk_dict)
    usage_numbers = extract_usage_numbers(usage_dict)
    return {
        "status": "success",
        "error_type": None,
        "error_message": None,
        "total_latency_sec": end - start,
        "ttfc_sec": None if first_chunk_at is None else first_chunk_at - start,
        "ttft_sec": None if first_content_at is None else first_content_at - start,
        "generation_after_first_token_sec": None if first_content_at is None else end - first_content_at,
        "chunks_seen": chunks_seen,
        "finish_reason": finish_reason,
        "answer": answer,
        "answer_preview": answer[:ANSWER_PREVIEW_CHARS],
        "answer_chars": len(answer),
        "answer_words": count_words(answer),
        "answer_hash": text_hash(answer),
        "reasoning_text": reasoning_text,
        "reasoning_chars": len(reasoning_text),
        "raw_usage_json": json.dumps(usage_dict, ensure_ascii=False),
        **usage_numbers,
    }


def run_payload_with_error_capture(*args, **kwargs) -> Dict[str, Any]:
    start = time.perf_counter()
    try:
        return run_payload(*args, **kwargs)
    except Exception as exc:
        return {
            "status": "error",
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "total_latency_sec": time.perf_counter() - start,
            "ttfc_sec": None,
            "ttft_sec": None,
            "generation_after_first_token_sec": None,
            "chunks_seen": 0,
            "finish_reason": None,
            "answer": "",
            "answer_preview": "",
            "answer_chars": 0,
            "answer_words": 0,
            "answer_hash": text_hash(""),
            "reasoning_text": "",
            "reasoning_chars": 0,
            "raw_usage_json": "{}",
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
            "precached_prompt_tokens": None,
            "reasoning_tokens": None,
        }


In [ ]:
# =========================
# 5. Discovery candidates
# =========================

BASE_DISCOVERY_PARAMS = {
    "temperature": 0.0,
    "max_tokens": 512,
}

# candidate_name должен быть стабильным: он попадет в Excel и логику выбора.
DISCOVERY_CANDIDATES: List[Dict[str, Any]] = [
    # Core/sampling controls
    {"name": "temperature_02", "category": "core", "patch": {"temperature": 0.2}},
    {"name": "temperature_05", "category": "core", "patch": {"temperature": 0.5}},
    {"name": "temperature_07", "category": "core", "patch": {"temperature": 0.7}},
    {"name": "top_p_08", "category": "core", "patch": {"top_p": 0.8}},
    {"name": "top_p_09", "category": "core", "patch": {"top_p": 0.9}},
    {"name": "top_p_095", "category": "core", "patch": {"top_p": 0.95}},
    {"name": "repetition_penalty_105", "category": "core", "patch": {"repetition_penalty": 1.05}},
    {"name": "max_tokens_2048", "category": "budget", "patch": {"max_tokens": 2048}},
    {"name": "max_tokens_4096", "category": "budget", "patch": {"max_tokens": 4096}},

    # Reasoning/thinking variants. Часть из них может не поддерживаться конкретным backend — это и выясняем.
    {"name": "reasoning_effort_none", "category": "reasoning", "mechanism": "reasoning_effort", "mode": "off", "patch": {"reasoning_effort": "none"}},
    {"name": "reasoning_effort_low", "category": "reasoning", "mechanism": "reasoning_effort", "mode": "on_low", "patch": {"reasoning_effort": "low"}},
    {"name": "reasoning_effort_medium", "category": "reasoning", "mechanism": "reasoning_effort", "mode": "on_medium", "patch": {"reasoning_effort": "medium"}},
    {"name": "reasoning_effort_high", "category": "reasoning", "mechanism": "reasoning_effort", "mode": "on_high", "patch": {"reasoning_effort": "high"}},

    {"name": "reasoning_obj_none", "category": "reasoning", "mechanism": "reasoning_object_effort", "mode": "off", "patch": {"reasoning": {"effort": "none"}}},
    {"name": "reasoning_obj_low", "category": "reasoning", "mechanism": "reasoning_object_effort", "mode": "on_low", "patch": {"reasoning": {"effort": "low"}}},
    {"name": "reasoning_obj_medium", "category": "reasoning", "mechanism": "reasoning_object_effort", "mode": "on_medium", "patch": {"reasoning": {"effort": "medium"}}},
    {"name": "reasoning_obj_high", "category": "reasoning", "mechanism": "reasoning_object_effort", "mode": "on_high", "patch": {"reasoning": {"effort": "high"}}},
    {"name": "reasoning_obj_enabled_false", "category": "reasoning", "mechanism": "reasoning_object_enabled", "mode": "off", "patch": {"reasoning": {"enabled": False}}},
    {"name": "reasoning_obj_enabled_true", "category": "reasoning", "mechanism": "reasoning_object_enabled", "mode": "on", "patch": {"reasoning": {"enabled": True}}},

    {"name": "enable_thinking_false", "category": "reasoning", "mechanism": "enable_thinking", "mode": "off", "patch": {"enable_thinking": False}},
    {"name": "enable_thinking_true", "category": "reasoning", "mechanism": "enable_thinking", "mode": "on", "patch": {"enable_thinking": True}},
    {"name": "thinking_enabled_false", "category": "reasoning", "mechanism": "thinking_enabled", "mode": "off", "patch": {"thinking": {"enabled": False}}},
    {"name": "thinking_enabled_true", "category": "reasoning", "mechanism": "thinking_enabled", "mode": "on", "patch": {"thinking": {"enabled": True}}},
    {"name": "thinking_budget_512", "category": "reasoning", "mechanism": "thinking_budget", "mode": "on_budget", "patch": {"thinking": {"budget_tokens": 512}}},
    {"name": "chat_template_thinking_false", "category": "reasoning", "mechanism": "chat_template_kwargs", "mode": "off", "patch": {"chat_template_kwargs": {"enable_thinking": False}}},
    {"name": "chat_template_thinking_true", "category": "reasoning", "mechanism": "chat_template_kwargs", "mode": "on", "patch": {"chat_template_kwargs": {"enable_thinking": True}}},
    {"name": "include_reasoning_true", "category": "reasoning_output", "mechanism": "include_reasoning", "mode": "include", "patch": {"include_reasoning": True}},
]

print("Discovery candidates:", len(DISCOVERY_CANDIDATES))


In [ ]:
# =========================
# 6. Fast capability discovery
# =========================


def candidate_family(candidate: Dict[str, Any]) -> str:
    if candidate.get("mechanism"):
        return str(candidate["mechanism"])
    patch = candidate.get("patch") or {}
    if len(patch) == 1:
        return next(iter(patch.keys()))
    return "mixed"


def make_capability_row() -> pd.Series:
    return pd.Series({
        "dataset_row_id": -1,
        "type": "technical_capability_probe",
        "stage": "technical_capability_probe",
        "question": CAPABILITY_USER_PROMPT,
        "system_prompt": CAPABILITY_SYSTEM_PROMPT,
        "user_prompt": CAPABILITY_USER_PROMPT,
        "system_chars": len(CAPABILITY_SYSTEM_PROMPT),
        "user_chars": len(CAPABILITY_USER_PROMPT),
        "input_chars": len(CAPABILITY_SYSTEM_PROMPT) + len(CAPABILITY_USER_PROMPT),
    })


def select_latency_discovery_rows(df: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for _, part in df.groupby("type", dropna=False):
        parts.append(part.sort_values("input_chars").head(LATENCY_DISCOVERY_ROWS_PER_STAGE))
    return pd.concat(parts, ignore_index=True).sort_values("input_chars").reset_index(drop=True)


def probe_one(model_key: str, model_name: str, row: pd.Series, candidate: Dict[str, Any], baseline_by_model: Dict[str, Dict[str, Any]], probe_kind: str) -> Dict[str, Any]:
    timeout = CAPABILITY_DISCOVERY_TIMEOUT_SEC if probe_kind == "capability" else LATENCY_DISCOVERY_TIMEOUT_SEC
    result = run_payload_with_error_capture(
        row["system_prompt"],
        row["user_prompt"],
        model_name,
        BASE_DISCOVERY_PARAMS,
        candidate.get("patch") or {},
        timeout_sec=timeout,
        use_streaming=False,
    )
    baseline = baseline_by_model.get((model_key, probe_kind, int(row["dataset_row_id"])), baseline_by_model.get((model_key, probe_kind), {}))
    accepted = result["status"] == "success"
    answer_changed = accepted and baseline.get("answer_hash") and result.get("answer_hash") != baseline.get("answer_hash")

    latency_changed = False
    if accepted and baseline.get("total_latency_sec") and result.get("total_latency_sec"):
        b = float(baseline["total_latency_sec"])
        r = float(result["total_latency_sec"])
        latency_changed = abs(r - b) / max(b, 0.001) >= 0.15

    length_changed = False
    if accepted and baseline.get("answer_chars") is not None:
        b = float(baseline.get("answer_chars") or 0)
        r = float(result.get("answer_chars") or 0)
        length_changed = abs(r - b) / max(b, 1.0) >= 0.10

    reasoning_observed = bool((result.get("reasoning_chars") or 0) > 0 or pd.notna(result.get("reasoning_tokens")))

    baseline_latency = baseline.get("total_latency_sec")
    result_latency = result.get("total_latency_sec")
    latency_delta_sec = None
    latency_delta_pct = None
    if accepted and baseline_latency and result_latency:
        latency_delta_sec = float(result_latency) - float(baseline_latency)
        latency_delta_pct = 100.0 * latency_delta_sec / max(float(baseline_latency), 0.001)

    baseline_answer_chars = baseline.get("answer_chars")
    result_answer_chars = result.get("answer_chars")
    answer_chars_delta = None
    answer_chars_delta_pct = None
    if accepted and baseline_answer_chars is not None and result_answer_chars is not None:
        answer_chars_delta = float(result_answer_chars or 0) - float(baseline_answer_chars or 0)
        answer_chars_delta_pct = 100.0 * answer_chars_delta / max(float(baseline_answer_chars or 0), 1.0)

    observed_effect = bool(answer_changed or latency_changed or length_changed or reasoning_observed)
    effective = accepted and (candidate.get("category") in {"core", "budget"} or observed_effect)

    record = {
        "probe_kind": probe_kind,
        "model_key": model_key,
        "model": model_name,
        "dataset_row_id": int(row["dataset_row_id"]),
        "type": row.get("type"),
        "input_chars": int(row.get("input_chars", 0)),
        "probe_name": candidate["name"],
        "parameter_family": candidate_family(candidate),
        "category": candidate.get("category"),
        "mechanism": candidate.get("mechanism"),
        "mode": candidate.get("mode"),
        "patch_json": json.dumps(candidate.get("patch") or {}, ensure_ascii=False, sort_keys=True),
        "accepted": accepted,
        "observed_effect": observed_effect,
        "effective": effective,
        "answer_changed_vs_baseline": answer_changed,
        "latency_changed_vs_baseline": latency_changed,
        "length_changed_vs_baseline": length_changed,
        "reasoning_observed": reasoning_observed,
        "baseline_latency_sec": baseline_latency,
        "latency_delta_sec": latency_delta_sec,
        "latency_delta_pct": latency_delta_pct,
        "abs_latency_delta_pct": None if latency_delta_pct is None else abs(latency_delta_pct),
        "baseline_answer_chars": baseline_answer_chars,
        "answer_chars_delta": answer_chars_delta,
        "answer_chars_delta_pct": answer_chars_delta_pct,
        "abs_answer_chars_delta_pct": None if answer_chars_delta_pct is None else abs(answer_chars_delta_pct),
    }
    record.update(result)
    return record


def build_available_parameters(df: pd.DataFrame) -> pd.DataFrame:
    capability = df[df["probe_kind"] == "capability"].copy()
    if capability.empty:
        return pd.DataFrame()
    rows = []
    for (model_key, family), part in capability.groupby(["model_key", "parameter_family"], dropna=False):
        accepted = part[part["accepted"] == True]
        rejected = part[part["accepted"] != True]
        rows.append({
            "model_key": model_key,
            "parameter_family": family,
            "category": ", ".join(sorted(set(str(x) for x in part["category"].dropna()))),
            "accepted_count": len(accepted),
            "rejected_count": len(rejected),
            "accepted_variants": ", ".join(accepted["probe_name"].tolist()),
            "accepted_patches": " | ".join(accepted["patch_json"].tolist()),
            "rejected_variants": ", ".join(rejected["probe_name"].tolist()),
            "observed_signal_variants": ", ".join(part[part["observed_effect"] == True]["probe_name"].tolist()),
        })
    return pd.DataFrame(rows).sort_values(["model_key", "category", "parameter_family"]).reset_index(drop=True)


if RUN_DISCOVERY:
    discovery_records = []
    baseline_by_model = {}
    capability_row = make_capability_row()

    for model_key, model_name in MODELS.items():
        baseline = run_payload_with_error_capture(
            capability_row["system_prompt"], capability_row["user_prompt"], model_name,
            BASE_DISCOVERY_PARAMS, {}, timeout_sec=CAPABILITY_DISCOVERY_TIMEOUT_SEC, use_streaming=False
        )
        baseline_by_model[(model_key, "capability")] = baseline
        baseline_record = {
            "probe_kind": "capability",
            "model_key": model_key,
            "model": model_name,
            "dataset_row_id": int(capability_row["dataset_row_id"]),
            "type": capability_row.get("type"),
            "input_chars": int(capability_row.get("input_chars", 0)),
            "probe_name": "baseline",
            "parameter_family": "baseline",
            "category": "baseline",
            "mechanism": None,
            "mode": None,
            "patch_json": "{}",
            "accepted": baseline["status"] == "success",
            "observed_effect": False,
            "effective": baseline["status"] == "success",
            "answer_changed_vs_baseline": False,
            "latency_changed_vs_baseline": False,
            "length_changed_vs_baseline": False,
            "reasoning_observed": bool((baseline.get("reasoning_chars") or 0) > 0 or pd.notna(baseline.get("reasoning_tokens"))),
            "baseline_latency_sec": None,
            "latency_delta_sec": None,
            "latency_delta_pct": None,
            "abs_latency_delta_pct": None,
            "baseline_answer_chars": None,
            "answer_chars_delta": None,
            "answer_chars_delta_pct": None,
            "abs_answer_chars_delta_pct": None,
        }
        baseline_record.update(baseline)
        discovery_records.append(baseline_record)

        for candidate in tqdm(DISCOVERY_CANDIDATES, desc=f"Capability {model_key}"):
            record = probe_one(model_key, model_name, capability_row, candidate, baseline_by_model, probe_kind="capability")
            discovery_records.append(record)
            with DISCOVERY_JSONL_PATH.open("a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

    df_discovery = pd.DataFrame(discovery_records)
else:
    if not DISCOVERY_JSONL_PATH.exists():
        raise ValueError("RUN_DISCOVERY=False, но discovery jsonl не найден")
    df_discovery = pd.read_json(DISCOVERY_JSONL_PATH, lines=True)

# Быстрый вывод: что backend реально принял.
df_available_parameters = build_available_parameters(df_discovery)
print("Capability discovery finished. Accepted parameters by model:")
display(df_available_parameters)

if STOP_AFTER_CAPABILITY_DISCOVERY:
    raise SystemExit("Stopped after capability discovery. Set STOP_AFTER_CAPABILITY_DISCOVERY=False to build configs and benchmark.")

# Опциональный дорогой latency-impact discovery на реальных prompts. По умолчанию выключен.
if RUN_LATENCY_IMPACT_DISCOVERY:
    latency_rows = select_latency_discovery_rows(df_inputs)
    print("Latency-impact discovery rows:", len(latency_rows))
    display(latency_rows[["dataset_row_id", "type", "question", "input_chars"]])

    accepted_candidates = [c for c in DISCOVERY_CANDIDATES if c["name"] in set(df_discovery[(df_discovery["probe_kind"] == "capability") & (df_discovery["accepted"] == True)]["probe_name"])]
    extra_records = []
    baseline_by_model = {}
    for model_key, model_name in MODELS.items():
        for _, row in latency_rows.iterrows():
            baseline = run_payload_with_error_capture(
                row["system_prompt"], row["user_prompt"], model_name,
                BASE_DISCOVERY_PARAMS, {}, timeout_sec=LATENCY_DISCOVERY_TIMEOUT_SEC, use_streaming=False
            )
            baseline_by_model[(model_key, "latency", int(row["dataset_row_id"]))] = baseline
            for candidate in tqdm(accepted_candidates, desc=f"Latency impact {model_key} row {int(row['dataset_row_id'])}"):
                rec = probe_one(model_key, model_name, row, candidate, baseline_by_model, probe_kind="latency")
                extra_records.append(rec)
                with DISCOVERY_JSONL_PATH.open("a", encoding="utf-8") as f:
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    if extra_records:
        df_discovery = pd.concat([df_discovery, pd.DataFrame(extra_records)], ignore_index=True)

print("Discovery result shape:", df_discovery.shape)
display(df_discovery.groupby(["probe_kind", "model_key", "category"]).agg(probes=("probe_name", "count"), accepted=("accepted", "sum"), effective=("effective", "sum"), observed_effect=("observed_effect", "sum")).reset_index())


In [ ]:
# =========================
# 7. Build latency-first adaptive configs from discovery
# =========================


def parse_patch_json(value: Any) -> Dict[str, Any]:
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    try:
        return json.loads(value)
    except Exception:
        return {}


def discovery_for_model(model_key: str) -> pd.DataFrame:
    return df_discovery[df_discovery["model_key"] == model_key].copy()


def probe_row(model_key: str, probe_name: str) -> Optional[pd.Series]:
    rows = df_discovery[(df_discovery["model_key"] == model_key) & (df_discovery["probe_name"] == probe_name)]
    if rows.empty:
        return None
    return rows.iloc[0]


def patch_from_probe(df: pd.DataFrame, model_key: str, probe_name: str) -> Optional[Dict[str, Any]]:
    rows = df[(df["model_key"] == model_key) & (df["probe_name"] == probe_name) & (df["accepted"] == True)]
    if rows.empty:
        return None
    return parse_patch_json(rows.iloc[0]["patch_json"])


def accepted_probe_names(df: pd.DataFrame, model_key: str) -> set:
    return set(df[(df["model_key"] == model_key) & (df["accepted"] == True)]["probe_name"].tolist())


def param_supported(df: pd.DataFrame, model_key: str, probe_name: str) -> bool:
    rows = df[(df["model_key"] == model_key) & (df["probe_name"] == probe_name)]
    return bool((not rows.empty) and rows.iloc[0]["accepted"])


def probe_latency_score(row: pd.Series) -> float:
    latency = row.get("abs_latency_delta_pct")
    answer = row.get("abs_answer_chars_delta_pct")
    score = 0.0
    if pd.notna(latency):
        score += float(latency)
    if pd.notna(answer):
        # Длина ответа тоже влияет на время, но слабее прямого latency-сигнала.
        score += 0.25 * float(answer)
    if bool(row.get("reasoning_observed", False)):
        score += 25.0
    return score


def build_parameter_impact_table() -> pd.DataFrame:
    df = df_discovery.copy()
    if "accepted" not in df.columns:
        return pd.DataFrame()
    df["latency_impact_score"] = df.apply(probe_latency_score, axis=1)
    cols = [
        "model_key", "probe_name", "category", "mechanism", "mode", "patch_json",
        "accepted", "observed_effect", "effective", "latency_delta_sec", "latency_delta_pct",
        "abs_latency_delta_pct", "answer_chars_delta_pct", "reasoning_observed", "latency_impact_score",
        "total_latency_sec", "answer_chars", "completion_tokens", "finish_reason", "error_type", "error_message",
    ]
    cols = [c for c in cols if c in df.columns]
    return df[cols].sort_values(["model_key", "accepted", "latency_impact_score"], ascending=[True, False, False]).reset_index(drop=True)


def select_reasoning_pair(df: pd.DataFrame, model_key: str) -> Optional[Dict[str, Any]]:
    preferred = [
        ("reasoning_effort", ["reasoning_effort_none"], ["reasoning_effort_medium", "reasoning_effort_low", "reasoning_effort_high"]),
        ("reasoning_object_effort", ["reasoning_obj_none"], ["reasoning_obj_medium", "reasoning_obj_low", "reasoning_obj_high"]),
        ("reasoning_object_enabled", ["reasoning_obj_enabled_false"], ["reasoning_obj_enabled_true"]),
        ("enable_thinking", ["enable_thinking_false"], ["enable_thinking_true"]),
        ("thinking_enabled", ["thinking_enabled_false"], ["thinking_enabled_true", "thinking_budget_512"]),
        ("chat_template_kwargs", ["chat_template_thinking_false"], ["chat_template_thinking_true"]),
    ]
    accepted = accepted_probe_names(df, model_key)
    candidates = []
    for mechanism, off_names, on_names in preferred:
        off_name = next((n for n in off_names if n in accepted), None)
        on_name = next((n for n in on_names if n in accepted), None)
        if not (off_name and on_name):
            continue
        pair_rows = df[(df["model_key"] == model_key) & (df["probe_name"].isin([off_name, on_name]))].copy()
        observed_cols = [c for c in ["observed_effect", "reasoning_observed"] if c in pair_rows.columns]
        observed_score = 0
        if observed_cols:
            observed_score = int(pair_rows[observed_cols].fillna(False).astype(bool).any(axis=1).sum())
        impact_score = float(pair_rows.apply(probe_latency_score, axis=1).sum())
        if REQUIRE_OBSERVED_EFFECT_FOR_REASONING and observed_score == 0:
            continue
        candidates.append({
            "mechanism": mechanism,
            "off_name": off_name,
            "on_name": on_name,
            "off_patch": patch_from_probe(df, model_key, off_name),
            "on_patch": patch_from_probe(df, model_key, on_name),
            "observed_score": observed_score,
            "impact_score": impact_score,
        })
    if not candidates:
        return None
    return sorted(candidates, key=lambda x: (x["observed_score"], x["impact_score"]), reverse=True)[0]


def clean_params(params: Dict[str, Any], model_key: str) -> Dict[str, Any]:
    out = dict(params)
    if "temperature" in out and not any(param_supported(df_discovery, model_key, p) for p in ["temperature_02", "temperature_05", "temperature_07"]):
        out.pop("temperature", None)
    if "top_p" in out and not any(param_supported(df_discovery, model_key, p) for p in ["top_p_08", "top_p_09", "top_p_095"]):
        out.pop("top_p", None)
    if "repetition_penalty" in out and not param_supported(df_discovery, model_key, "repetition_penalty_105"):
        out.pop("repetition_penalty", None)
    if "max_tokens" in out:
        wanted = int(out["max_tokens"])
        if wanted == 2048 and not param_supported(df_discovery, model_key, "max_tokens_2048"):
            out["max_tokens"] = 4096 if param_supported(df_discovery, model_key, "max_tokens_4096") else None
        if wanted == 4096 and not param_supported(df_discovery, model_key, "max_tokens_4096"):
            out["max_tokens"] = 2048 if param_supported(df_discovery, model_key, "max_tokens_2048") else None
        if out.get("max_tokens") is None:
            out.pop("max_tokens", None)
    return out


def sanitize_profile(name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", name).strip("_").lower()


def make_config(model_key: str, model_name: str, profile: str, base_params: Dict[str, Any], extra_patch: Optional[Dict[str, Any]], why: str, source_probes: Optional[List[str]] = None) -> Dict[str, Any]:
    params = clean_params(base_params, model_key)
    extra_patch = extra_patch or {}
    cfg = {
        "config_id": f"{model_key}_{profile}",
        "model_key": model_key,
        "model": model_name,
        "profile": profile,
        "max_tokens": params.get("max_tokens"),
        "temperature": params.get("temperature"),
        "top_p": params.get("top_p"),
        "repetition_penalty": params.get("repetition_penalty"),
        "extra_params_json": json.dumps(extra_patch, ensure_ascii=False, sort_keys=True),
        "source_probes": ", ".join(source_probes or []),
        "why": why,
    }
    return cfg


def top_latency_sampling_patches(model_key: str, limit: int = 3) -> List[Dict[str, Any]]:
    df = discovery_for_model(model_key)
    df = df[(df["accepted"] == True) & (df["category"].isin(["core"]))].copy()
    if df.empty:
        return []
    df["latency_impact_score"] = df.apply(probe_latency_score, axis=1)
    df = df.sort_values("latency_impact_score", ascending=False)
    selected = []
    seen_keys = set()
    for _, row in df.iterrows():
        patch = parse_patch_json(row["patch_json"])
        keys = tuple(sorted(patch.keys()))
        # Берем разные типы параметров, чтобы не получить три temperature подряд.
        if keys in seen_keys:
            continue
        seen_keys.add(keys)
        selected.append({
            "probe_name": row["probe_name"],
            "patch": patch,
            "impact_score": float(row["latency_impact_score"]),
            "latency_delta_pct": row.get("latency_delta_pct"),
        })
        if len(selected) >= limit:
            break
    return selected


def merge_core_params(base: Dict[str, Any], patch: Dict[str, Any]) -> Dict[str, Any]:
    merged = dict(base)
    for k, v in patch.items():
        if k in {"temperature", "top_p", "repetition_penalty", "max_tokens"}:
            merged[k] = v
    return merged


def build_configs_for_model(model_key: str, model_name: str) -> List[Dict[str, Any]]:
    rp = select_reasoning_pair(df_discovery, model_key)
    top_sampling = top_latency_sampling_patches(model_key, limit=3)

    stable = {"max_tokens": 4096, "temperature": 0.2, "top_p": 0.9}
    stable_2048 = {**stable, "max_tokens": 2048}
    configs = []

    if rp:
        configs.append(make_config(model_key, model_name, "latency_reasoning_off_stable_4096", stable, rp["off_patch"], f"Контрольный быстрый/стабильный режим: reasoning OFF через {rp['mechanism']}", [rp["off_name"]]))
        configs.append(make_config(model_key, model_name, "latency_reasoning_on_stable_4096", stable, rp["on_patch"], f"Пара к OFF: reasoning ON через {rp['mechanism']}", [rp["on_name"]]))
        configs.append(make_config(model_key, model_name, "latency_reasoning_off_stable_2048", stable_2048, rp["off_patch"], "Проверка влияния max_tokens=2048 на время и обрезания при reasoning OFF", ["max_tokens_2048", rp["off_name"]]))
        configs.append(make_config(model_key, model_name, "latency_reasoning_on_stable_2048", stable_2048, rp["on_patch"], "Проверка max_tokens=2048 при reasoning ON", ["max_tokens_2048", rp["on_name"]]))
        # Последние два слота отдаем тем sampling-параметрам, которые discovery сильнее всего шатнули latency.
        for item in top_sampling[:2]:
            params = merge_core_params(stable, item["patch"])
            profile = f"latency_{sanitize_profile(item['probe_name'])}_reasoning_on_4096"
            why = f"Sampling-параметр из discovery с высоким latency impact ({item['impact_score']:.1f}); reasoning ON"
            configs.append(make_config(model_key, model_name, profile, params, rp["on_patch"], why, [item["probe_name"], rp["on_name"]]))
    else:
        configs.append(make_config(model_key, model_name, "latency_stable_4096", stable, {}, "Контрольный стабильный режим; reasoning-пара не найдена"))
        configs.append(make_config(model_key, model_name, "latency_stable_2048", stable_2048, {}, "Проверка влияния max_tokens=2048 на время и обрезания", ["max_tokens_2048"]))
        for item in top_sampling[:4]:
            params = merge_core_params(stable, item["patch"])
            profile = f"latency_{sanitize_profile(item['probe_name'])}_4096"
            why = f"Параметр выбран потому, что discovery показал высокий latency impact ({item['impact_score']:.1f})"
            configs.append(make_config(model_key, model_name, profile, params, {}, why, [item["probe_name"]]))

    unique = []
    seen = set()
    for cfg in configs:
        sig = (cfg["model"], cfg.get("max_tokens"), cfg.get("temperature"), cfg.get("top_p"), cfg.get("repetition_penalty"), cfg.get("extra_params_json"))
        if sig not in seen:
            seen.add(sig)
            unique.append(cfg)
    return unique[:MAX_CONFIGS_PER_MODEL]


df_parameter_impact = build_parameter_impact_table()
configs = []
reasoning_pairs = []
for model_key, model_name in MODELS.items():
    rp = select_reasoning_pair(df_discovery, model_key)
    reasoning_pairs.append({"model_key": model_key, **(rp or {"mechanism": None, "off_name": None, "on_name": None, "off_patch": None, "on_patch": None, "observed_score": 0, "impact_score": 0})})
    configs.extend(build_configs_for_model(model_key, model_name))

configs = configs[:MAX_TOTAL_CONFIGS]
df_configs = pd.DataFrame(configs)
df_reasoning_pairs = pd.DataFrame(reasoning_pairs)

print("Final configs:", len(df_configs))
print("Selection principle: latency-first from discovery; mandatory max_tokens/reasoning controls when available.")
display(df_reasoning_pairs)
display(df_parameter_impact.head(30))
display(df_configs)


In [ ]:
# =========================
# 8. Select benchmark rows
# =========================


def select_rows(df: pd.DataFrame) -> pd.DataFrame:
    if RUN_MODE == "full":
        selected = df.copy()
    elif RUN_MODE == "smoke":
        selected = pd.concat([p.sort_values("input_chars").head(SMOKE_ROWS_PER_STAGE) for _, p in df.groupby("type", dropna=False)], ignore_index=True)
    elif RUN_MODE == "sample":
        selected = df.sample(min(SAMPLE_ROWS_TOTAL, len(df)), random_state=RANDOM_SEED).copy()
    else:
        raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")
    return selected.sort_values(["type", "input_chars", "dataset_row_id"]).reset_index(drop=True)

df_run = select_rows(df_inputs)
print(f"RUN_MODE={RUN_MODE}; rows={len(df_run)}; configs={len(df_configs)}; expected calls={len(df_run) * len(df_configs)}")
display(df_run.groupby("type").agg(rows=("dataset_row_id", "count"), min_chars=("input_chars", "min"), max_chars=("input_chars", "max")).reset_index())


In [ ]:
# =========================
# 9. Benchmark runner
# =========================


def config_base_params(cfg: Dict[str, Any]) -> Dict[str, Any]:
    params = {}
    for k in ["max_tokens", "temperature", "top_p", "repetition_penalty"]:
        v = cfg.get(k)
        if pd.notna(v):
            if k == "max_tokens":
                v = int(v)
            params[k] = v
    return params


def config_extra_params(cfg: Dict[str, Any]) -> Dict[str, Any]:
    raw = cfg.get("extra_params_json") or "{}"
    try:
        return json.loads(raw)
    except Exception:
        return {}


def run_with_retries(row: pd.Series, cfg: Dict[str, Any]) -> Dict[str, Any]:
    last = None
    for attempt in range(1, MAX_RETRIES + 2):
        result = run_payload_with_error_capture(
            row["system_prompt"], row["user_prompt"], cfg["model"],
            config_base_params(cfg), config_extra_params(cfg),
            timeout_sec=REQUEST_TIMEOUT_SEC, use_streaming=USE_STREAMING,
        )
        result["attempts"] = attempt
        if result["status"] == "success":
            return result
        last = result
        if attempt <= MAX_RETRIES:
            time.sleep(RETRY_BACKOFF_SEC * attempt + random.random() * JITTER_SEC)
    return last


def ordered_configs_for_question(question_order_index: int, dataset_row_id: int) -> List[Dict[str, Any]]:
    cfgs = [r.to_dict() for _, r in df_configs.iterrows()]
    if CONFIG_ORDER_STRATEGY == "fixed":
        return cfgs
    if CONFIG_ORDER_STRATEGY == "rotate":
        shift = question_order_index % len(cfgs)
        return cfgs[shift:] + cfgs[:shift]
    if CONFIG_ORDER_STRATEGY == "shuffle_by_question":
        rng = random.Random(RANDOM_SEED + int(dataset_row_id))
        rng.shuffle(cfgs)
        return cfgs
    raise ValueError(CONFIG_ORDER_STRATEGY)


def build_jobs() -> List[Dict[str, Any]]:
    completed = set()
    if RAW_JSONL_PATH.exists():
        with RAW_JSONL_PATH.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    item = json.loads(line)
                    completed.add((int(item["dataset_row_id"]), item["config_id"]))
                except Exception:
                    pass
    jobs = []
    global_order = 0
    for q_idx, (_, row) in enumerate(df_run.iterrows()):
        for c_idx, cfg in enumerate(ordered_configs_for_question(q_idx, int(row["dataset_row_id"]))):
            key = (int(row["dataset_row_id"]), cfg["config_id"])
            if key not in completed:
                jobs.append({"row": row, "cfg": cfg, "question_order_index": q_idx, "config_order_within_question": c_idx, "global_job_order": global_order})
            global_order += 1
    return jobs


def build_record(job: Dict[str, Any], result: Dict[str, Any]) -> Dict[str, Any]:
    row, cfg = job["row"], job["cfg"]
    rec = {
        "run_started_at": RUN_STARTED_AT,
        "run_mode": RUN_MODE,
        "config_order_strategy": CONFIG_ORDER_STRATEGY,
        "global_job_order": int(job["global_job_order"]),
        "question_order_index": int(job["question_order_index"]),
        "config_order_within_question": int(job["config_order_within_question"]),
        "dataset_row_id": int(row["dataset_row_id"]),
        "question_run_id": row.get("question_run_id"),
        "type": row.get("type"),
        "stage": row.get("stage"),
        "question": row.get("question"),
        "source_file": row.get("source_file"),
        "source_line": row.get("source_line"),
        "system_chars": int(row.get("system_chars", 0)),
        "user_chars": int(row.get("user_chars", 0)),
        "input_chars": int(row.get("input_chars", 0)),
        "old_output_chars": int(row.get("old_output_chars", 0)),
    }
    for k in ["config_id", "model_key", "model", "profile", "max_tokens", "temperature", "top_p", "repetition_penalty", "extra_params_json", "why"]:
        rec[k] = cfg.get(k)
    rec.update(result)
    rec["empty_answer"] = (rec.get("answer_chars") or 0) == 0
    rec["possibly_truncated"] = str(rec.get("finish_reason", "")).lower() in {"length", "max_tokens", "token_limit"} or (
        pd.notna(rec.get("completion_tokens")) and pd.notna(rec.get("max_tokens")) and float(rec.get("completion_tokens")) >= 0.95 * float(rec.get("max_tokens"))
    )
    rec["ok_nonempty"] = rec.get("status") == "success" and not rec["empty_answer"]
    rec["error_or_empty"] = not rec["ok_nonempty"]
    if not SAVE_FULL_ANSWERS_TO_EXCEL:
        rec["answer"] = ""
    return rec


def execute_job(job: Dict[str, Any]) -> Dict[str, Any]:
    return build_record(job, run_with_retries(job["row"], job["cfg"]))


In [ ]:
# =========================
# 10. Execute benchmark with checkpoint Excel every N questions
# =========================


def read_raw_jsonl(path: Path) -> pd.DataFrame:
    records = []
    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        records.append(json.loads(line))
                    except Exception:
                        pass
    return pd.DataFrame(records)


def lightweight_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    for col in ["total_latency_sec", "ttft_sec", "completion_tokens", "total_tokens", "answer_words", "max_tokens"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "ok_nonempty" not in df.columns:
        df["ok_nonempty"] = (df.get("status") == "success") & (~df.get("empty_answer", False).fillna(False))
    if "possibly_truncated" not in df.columns:
        df["possibly_truncated"] = False
    return df.groupby(["config_id", "model_key"], dropna=False).agg(
        calls=("dataset_row_id", "count"),
        success_nonempty=("ok_nonempty", "sum"),
        avg_total_latency_sec=("total_latency_sec", "mean"),
        p90_total_latency_sec=("total_latency_sec", lambda s: pd.to_numeric(s, errors="coerce").quantile(.9)),
        avg_ttft_sec=("ttft_sec", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_answer_words=("answer_words", "mean"),
        possible_truncations=("possibly_truncated", "sum"),
    ).reset_index()


def write_checkpoint_excel(completed_question_groups: int, reason: str = "periodic") -> Path:
    current_raw = read_raw_jsonl(RAW_JSONL_PATH)
    checkpoint_readme = pd.DataFrame([
        ["run_started_at", RUN_STARTED_AT],
        ["checkpoint_reason", reason],
        ["completed_question_groups_in_this_session", completed_question_groups],
        ["checkpoint_every_questions", CHECKPOINT_EVERY_QUESTIONS],
        ["raw_records_saved", len(current_raw)],
        ["actual_configs", len(df_configs)],
        ["max_total_configs", MAX_TOTAL_CONFIGS],
        ["selection_principle", "latency-first from discovery"],
        ["job_order", "question-major"],
        ["raw_jsonl", str(RAW_JSONL_PATH)],
    ], columns=["field", "value"])

    partial_summary = lightweight_summary(current_raw.copy()) if not current_raw.empty else pd.DataFrame()
    numbered_path = OUTPUT_DIR / f"checkpoint_after_{completed_question_groups:04d}_questions.xlsx"
    target_path = numbered_path if SAVE_NUMBERED_CHECKPOINTS else CHECKPOINT_LATEST_PATH

    with pd.ExcelWriter(target_path, engine="xlsxwriter") as writer:
        checkpoint_readme.to_excel(writer, sheet_name="README", index=False)
        df_configs.to_excel(writer, sheet_name="Configs", index=False)
        df_available_parameters.to_excel(writer, sheet_name="AvailableParameters", index=False)
        df_discovery.to_excel(writer, sheet_name="DiscoveryProbes", index=False)
        df_parameter_impact.to_excel(writer, sheet_name="ParameterImpact", index=False)
        df_reasoning_pairs.to_excel(writer, sheet_name="ReasoningPairs", index=False)
        partial_summary.to_excel(writer, sheet_name="PartialSummary", index=False)
        current_raw.to_excel(writer, sheet_name="RawResults", index=False)
        workbook = writer.book
        header_fmt = workbook.add_format({"bold": True, "bg_color": "#D9EAF7", "border": 1})
        wrap_fmt = workbook.add_format({"text_wrap": True, "valign": "top"})
        for sheet_name in writer.sheets:
            ws = writer.sheets[sheet_name]
            ws.freeze_panes(1, 0)
            ws.set_row(0, None, header_fmt)
            ws.set_column(0, 12, 20)
            ws.set_column(12, 60, 28, wrap_fmt)

    if SAVE_NUMBERED_CHECKPOINTS:
        import shutil
        shutil.copyfile(target_path, CHECKPOINT_LATEST_PATH)
    print(f"Checkpoint Excel saved: {target_path}")
    return target_path


def group_jobs_by_question(jobs: List[Dict[str, Any]]) -> List[List[Dict[str, Any]]]:
    groups = []
    current_q = None
    batch = []
    for job in jobs:
        q = job["question_order_index"]
        if current_q is None:
            current_q = q
        if q != current_q:
            groups.append(batch)
            batch = []
            current_q = q
        batch.append(job)
    if batch:
        groups.append(batch)
    return groups


if RUN_BENCHMARK:
    jobs = build_jobs()
    question_batches = group_jobs_by_question(jobs)
    print("Jobs to run:", len(jobs))
    print("Question groups to run:", len(question_batches))
    print("Job order: question-major; each prompt runs across all configs before the next prompt.")
    print(f"Checkpoint Excel every {CHECKPOINT_EVERY_QUESTIONS} question groups.")
    all_new = []
    completed_question_groups = 0
    with RAW_JSONL_PATH.open("a", encoding="utf-8") as out_f:
        pbar = tqdm(total=len(jobs), desc="Benchmark calls")
        for batch in question_batches:
            if MAX_WORKERS == 1:
                batch_records = [execute_job(job) for job in batch]
            else:
                with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                    futures = [executor.submit(execute_job, job) for job in batch]
                    batch_records = [fut.result() for fut in as_completed(futures)]
            # Пишем пачку вопроса целиком, чтобы checkpoint никогда не ловил половину вопроса.
            for rec in batch_records:
                out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                out_f.flush()
                all_new.append(rec)
                pbar.update(1)
            completed_question_groups += 1
            if CHECKPOINT_EVERY_QUESTIONS and completed_question_groups % CHECKPOINT_EVERY_QUESTIONS == 0:
                write_checkpoint_excel(completed_question_groups, reason="periodic")
        pbar.close()
    if all_new:
        write_checkpoint_excel(completed_question_groups, reason="final_after_benchmark_cell")
    print("New records:", len(all_new))
else:
    print("RUN_BENCHMARK=False: benchmark skipped")



In [ ]:
# =========================
# 11. Load results and summarize
# =========================

records = []
if RAW_JSONL_PATH.exists():
    with RAW_JSONL_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
if not records:
    raise ValueError("Нет raw_results.jsonl. Запустите benchmark или проверьте путь.")

df_results = pd.DataFrame(records)
for col in ["total_latency_sec", "ttft_sec", "completion_tokens", "prompt_tokens", "total_tokens", "reasoning_tokens", "answer_chars", "answer_words", "input_chars", "max_tokens"]:
    if col in df_results.columns:
        df_results[col] = pd.to_numeric(df_results[col], errors="coerce")


def q(s, p):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return np.nan if s.empty else float(s.quantile(p))

summary = df_results.groupby(["config_id", "model_key", "model", "profile", "max_tokens", "temperature", "top_p", "repetition_penalty", "extra_params_json"], dropna=False).agg(
    calls=("dataset_row_id", "count"),
    success_nonempty=("ok_nonempty", "sum"),
    errors_or_empty=("error_or_empty", "sum"),
    empty_answers=("empty_answer", "sum"),
    possible_truncations=("possibly_truncated", "sum"),
    avg_total_latency_sec=("total_latency_sec", "mean"),
    p50_total_latency_sec=("total_latency_sec", lambda s: q(s, .5)),
    p90_total_latency_sec=("total_latency_sec", lambda s: q(s, .9)),
    avg_ttft_sec=("ttft_sec", "mean"),
    avg_completion_tokens=("completion_tokens", "mean"),
    avg_total_tokens=("total_tokens", "mean"),
    avg_reasoning_tokens=("reasoning_tokens", "mean"),
    avg_answer_chars=("answer_chars", "mean"),
    avg_answer_words=("answer_words", "mean"),
    avg_reasoning_chars=("reasoning_chars", "mean"),
).reset_index()
summary["success_rate_pct"] = 100 * summary["success_nonempty"] / summary["calls"]
summary["empty_answer_rate_pct"] = 100 * summary["empty_answers"] / summary["calls"]
summary["possible_truncation_rate_pct"] = 100 * summary["possible_truncations"] / summary["calls"]
summary = summary.round(3)

by_stage = df_results.groupby(["config_id", "model_key", "type"], dropna=False).agg(
    calls=("dataset_row_id", "count"),
    success_nonempty=("ok_nonempty", "sum"),
    avg_total_latency_sec=("total_latency_sec", "mean"),
    p90_total_latency_sec=("total_latency_sec", lambda s: q(s, .9)),
    avg_answer_words=("answer_words", "mean"),
    possible_truncations=("possibly_truncated", "sum"),
    empty_answers=("empty_answer", "sum"),
).reset_index()
by_stage["success_rate_pct"] = 100 * by_stage["success_nonempty"] / by_stage["calls"]
by_stage["possible_truncation_rate_pct"] = 100 * by_stage["possible_truncations"] / by_stage["calls"]
by_stage["empty_answer_rate_pct"] = 100 * by_stage["empty_answers"] / by_stage["calls"]
by_stage = by_stage.round(3)

rank = summary.copy()
rank["latency_rank"] = rank["p90_total_latency_sec"].rank(method="min", ascending=True)
rank["success_rank"] = rank["success_rate_pct"].rank(method="min", ascending=False)
rank["truncation_rank"] = rank["possible_truncation_rate_pct"].rank(method="min", ascending=True)
rank["empty_rank"] = rank["empty_answer_rate_pct"].rank(method="min", ascending=True)
rank["rank_score"] = 0.4 * rank["success_rank"] + 0.3 * rank["latency_rank"] + 0.2 * rank["truncation_rank"] + 0.1 * rank["empty_rank"]
rank = rank.sort_values(["rank_score", "success_rate_pct", "p90_total_latency_sec"], ascending=[True, False, True]).reset_index(drop=True)

display(rank.head(12))


In [ ]:
# =========================
# 12. Save Excel report
# =========================

readme = pd.DataFrame([
    ["run_started_at", RUN_STARTED_AT],
    ["input_csv", str(DATA_CSV)],
    ["run_mode", RUN_MODE],
    ["models", ", ".join(MODELS.values())],
    ["max_total_configs", MAX_TOTAL_CONFIGS],
    ["actual_configs", len(df_configs)],
    ["discovery_first", "yes"],
    ["capability_discovery_prompt", CAPABILITY_USER_PROMPT],
    ["run_latency_impact_discovery", RUN_LATENCY_IMPACT_DISCOVERY],
    ["selection_principle", "available-parameters first; latency-oriented heuristic/impact; mandatory max_tokens/reasoning controls when available"],
    ["checkpoint_every_questions", CHECKPOINT_EVERY_QUESTIONS],
    ["checkpoint_latest", str(CHECKPOINT_LATEST_PATH)],
    ["job_order", "question-major"],
    ["note", "Final configs are built from accepted discovery probes, prioritized by observed latency impact"],
], columns=["field", "value"])

raw_excel = df_results.copy()
if "answer" in raw_excel.columns:
    raw_excel["answer"] = raw_excel["answer"].fillna("").str.slice(0, 30000)
if "reasoning_text" in raw_excel.columns:
    raw_excel["reasoning_text"] = raw_excel["reasoning_text"].fillna("").str.slice(0, 5000)

with pd.ExcelWriter(EXCEL_REPORT_PATH, engine="xlsxwriter") as writer:
    readme.to_excel(writer, sheet_name="README", index=False)
    df_available_parameters.to_excel(writer, sheet_name="AvailableParameters", index=False)
    df_discovery.to_excel(writer, sheet_name="DiscoveryProbes", index=False)
    df_parameter_impact.to_excel(writer, sheet_name="ParameterImpact", index=False)
    df_reasoning_pairs.to_excel(writer, sheet_name="ReasoningPairs", index=False)
    df_configs.to_excel(writer, sheet_name="Configs", index=False)
    df_run.to_excel(writer, sheet_name="InputRows", index=False)
    rank.to_excel(writer, sheet_name="RankedConfigs", index=False)
    summary.to_excel(writer, sheet_name="SummaryOverall", index=False)
    by_stage.to_excel(writer, sheet_name="SummaryByStage", index=False)
    raw_excel.to_excel(writer, sheet_name="RawResults", index=False)

    workbook = writer.book
    header_fmt = workbook.add_format({"bold": True, "bg_color": "#D9EAF7", "border": 1})
    wrap_fmt = workbook.add_format({"text_wrap": True, "valign": "top"})
    for sheet_name in writer.sheets:
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, 100000, 30)
        ws.set_row(0, None, header_fmt)
        ws.set_column(0, 0, 24)
        ws.set_column(1, 10, 18)
        ws.set_column(10, 20, 22)
        ws.set_column(20, 60, 28, wrap_fmt)

print("Excel report:", EXCEL_REPORT_PATH.resolve())


